# Índice de un MW, v0 (17 oct 2026)

**Pregunta:** cuánto rinde un megavatio por día en tres usos: minar Bitcoin, alojar GPUs de un tercero, o rentar H100 propias. Y de paso: ¿hay algo que correlacionar entre el fee rate de Bitcoin y el precio de la inferencia?

**Datos:** `data/*.csv`, cada fila con su URL. Huecos reales, sin rellenar. Hashprice y BTC casi completos de ago-2024 a sep-2026; H100 solo puntos; fees solo puntos; tokens solo puntos.

**Supuestos:** en el dict `SUPUESTOS`. Cambiar ahí, no en las fórmulas.

**Verificación de cordura:** con hashprice de jun-2026 (US$30.37) y flota 17.5 J/TH, el modelo da ~US$1,735 bruto por MW/día. Luxor publica para el bucket 14-19 J/TH ~US$77/MWh = US$1,848/MW/día. Está dentro del rango, y la diferencia es que el bucket de Luxor incluye flotas más eficientes.

In [1]:
"""
Índice de un MW, v0.
Compara cuánto rinde un megavatio por día en tres usos:
  1. Minar Bitcoin (hashprice x eficiencia)
  2. Alojar GPUs de un tercero (hosting, sin capex de GPU)
  3. Alquilar H100 propias (índice H100 x densidad x utilización)
Más: fee rate y precio por token, para ver si hay algo que correlacionar.

Todo supuesto está en SUPUESTOS. Cambiarlos cambia el número; no hay magia.
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path(".") / "data"
OUT = Path(".") / "out"
OUT.mkdir(exist_ok=True)

# ---------- SUPUESTOS (editar aquí) ----------
SUPUESTOS = {
    # minería
    "j_por_th_flota_vieja": 29.5,   # S19j Pro
    "j_por_th_flota_media": 17.5,   # S21
    "j_por_th_flota_nueva": 13.5,   # S21 XP
    "usd_kwh": 0.048,               # Luxor jul 2026
    # GPU propia
    "h100_por_mw_it": 740,          # medio del rango 690-785
    "pue": 1.3,                     # aire eficiente / líquida mediocre
    "utilizacion": 0.71,            # Epoch
    # hosting
    "hosting_usd_mw_it_anio": 1_850_000,  # TeraWulf-Fluidstack
    # capex (para la columna de payback, solo referencia)
    "capex_mina_usd_mw": 850_000,
    "capex_ia_usd_mw": 20_000_000,
}


def ingreso_mina_usd_mw_dia(hashprice_usd_ph_dia, j_por_th):
    """1 MW alimentando ASICs de j_por_th J/TH durante un día.
    TH/s que caben en 1 MW = 1e6 W / (j_por_th W/TH) ; en PH/s = /1000.
    Ingreso bruto = PH/s x hashprice (USD/PH/día)."""
    ph_s = (1e6 / j_por_th) / 1000
    bruto = ph_s * hashprice_usd_ph_dia
    electricidad = 1000 * 24 * SUPUESTOS["usd_kwh"]  # 1 MW x 24 h
    return bruto, bruto - electricidad


def ingreso_gpu_usd_mw_dia(h100_usd_hora):
    """1 MW de capacidad IT llena de H100 propias.
    Con PUE, el MW de red alimenta 1/PUE MW IT."""
    mw_it = 1 / SUPUESTOS["pue"]
    gpus = SUPUESTOS["h100_por_mw_it"] * mw_it
    bruto = gpus * h100_usd_hora * 24 * SUPUESTOS["utilizacion"]
    electricidad = 1000 * 24 * SUPUESTOS["usd_kwh"]
    return bruto, bruto - electricidad


def ingreso_hosting_usd_mw_dia():
    mw_it = 1 / SUPUESTOS["pue"]
    return SUPUESTOS["hosting_usd_mw_it_anio"] * mw_it / 365


def cargar():
    hp = pd.read_csv(DATA / "hashprice.csv")
    btc = pd.read_csv(DATA / "btc.csv")
    h100 = pd.read_csv(DATA / "h100.csv")
    fees = pd.read_csv(DATA / "fees.csv")
    tok = pd.read_csv(DATA / "tokens.csv")
    for d in (hp, btc, h100, fees, tok):
        d["mes"] = pd.PeriodIndex(d["mes"], freq="M")
    # H100: si hay varios puntos en un mes, promedio
    h100m = h100.groupby("mes", as_index=False)["h100_usd_hora"].mean()
    df = (hp[["mes", "hashprice_usd_ph_dia", "tipo"]]
          .merge(btc[["mes", "btc_cierre_usd"]], on="mes", how="outer")
          .merge(h100m, on="mes", how="outer")
          .merge(fees[["mes", "fees_pct_ingreso_minero"]], on="mes", how="outer")
          .sort_values("mes").reset_index(drop=True))
    return df, tok


def construir(df):
    for nombre, j in [("vieja", SUPUESTOS["j_por_th_flota_vieja"]),
                      ("media", SUPUESTOS["j_por_th_flota_media"]),
                      ("nueva", SUPUESTOS["j_por_th_flota_nueva"])]:
        b, n = zip(*[ingreso_mina_usd_mw_dia(h, j) if pd.notna(h) else (np.nan, np.nan)
                     for h in df["hashprice_usd_ph_dia"]])
        df[f"mina_{nombre}_bruto"] = b
        df[f"mina_{nombre}_neto"] = n
    b, n = zip(*[ingreso_gpu_usd_mw_dia(h) if pd.notna(h) else (np.nan, np.nan)
                 for h in df["h100_usd_hora"]])
    df["gpu_bruto"] = b
    df["gpu_neto"] = n
    df["hosting"] = ingreso_hosting_usd_mw_dia()
    # la brecha que importa: cuántas veces rinde el MW en GPU vs minando con flota media
    df["brecha_gpu_vs_mina_media"] = df["gpu_neto"] / df["mina_media_neto"]
    df["brecha_hosting_vs_mina_media"] = df["hosting"] / df["mina_media_neto"]
    return df


def graficar(df, tok):
    # reindexar a todos los meses para que los huecos rompan la línea en vez de interpolarse
    full = pd.period_range(df["mes"].min(), df["mes"].max(), freq="M")
    d = df.set_index("mes").reindex(full).reset_index().rename(columns={"index": "mes"})
    x = d["mes"].dt.to_timestamp()

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(x, d["mina_nueva_neto"], label="Minar, flota 13.5 J/TH (neto de luz)", lw=2)
    ax.plot(x, d["mina_media_neto"], label="Minar, flota 17.5 J/TH (neto de luz)", lw=2)
    ax.plot(x, d["mina_vieja_neto"], label="Minar, flota 29.5 J/TH (neto de luz)", lw=2)
    ax.axhline(d["hosting"].iloc[-1], color="C3", ls="--", lw=2, label="Hosting de IA (TeraWulf/Fluidstack, sin GPU propia)")
    g = d.dropna(subset=["gpu_neto"])
    ax.scatter(g["mes"].dt.to_timestamp(), g["gpu_neto"], label="H100 propias (puntos con dato)", zorder=5, s=40)
    ax.axhline(0, color="k", lw=.8)
    ax.set_yscale("symlog", linthresh=500)
    ax.set_ylabel("USD por MW por día (escala symlog)")
    ax.set_title("Qué rinde un megavatio: minar vs alojar IA vs rentar H100")
    ax.grid(alpha=.3, which="both")
    ax.legend(fontsize=8, loc="upper left")
    fig.tight_layout()
    fig.savefig(OUT / "mw_minar_vs_ia.png", dpi=150)

    # fee share y token price: la "correlación" que no existe
    fig, ax1 = plt.subplots(figsize=(11, 5))
    f = df.dropna(subset=["fees_pct_ingreso_minero"])
    ax1.bar(f["mes"].dt.to_timestamp(), f["fees_pct_ingreso_minero"], width=20, label="Fees como % del ingreso minero")
    ax1.set_yscale("log")
    ax1.set_ylabel("% del ingreso minero (log)")
    ax2 = ax1.twinx()
    t4 = tok[tok["nivel"].str.startswith("GPT-4")]
    t35 = tok[tok["nivel"].str.startswith("GPT-3.5")]
    ax2.plot(t4["mes"].dt.to_timestamp(), t4["usd_por_millon"], "o-", color="C3", label="USD/M tokens, nivel GPT-4")
    ax2.plot(t35["mes"].dt.to_timestamp(), t35["usd_por_millon"], "s-", color="C2", label="USD/M tokens, nivel GPT-3.5")
    ax2.set_yscale("log")
    ax2.set_ylabel("USD por millón de tokens (log)")
    ax1.set_title("Fee rate de Bitcoin vs precio de inferencia: dos cosas que no se tocan")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper right")
    fig.tight_layout()
    fig.savefig(OUT / "fees_vs_tokens.png", dpi=150)


def correlaciones(df):
    """Pearson sobre meses donde ambos existen. Pocos puntos: no es prueba, es rastro."""
    out = {}
    pares = [("hashprice_usd_ph_dia", "h100_usd_hora"),
             ("btc_cierre_usd", "h100_usd_hora"),
             ("hashprice_usd_ph_dia", "fees_pct_ingreso_minero"),
             ("btc_cierre_usd", "hashprice_usd_ph_dia")]
    for a, b in pares:
        s = df[[a, b]].dropna()
        out[f"{a} ~ {b}"] = (round(s[a].corr(s[b]), 2) if len(s) > 2 else np.nan, len(s))
    return out


if True:
    df, tok = cargar()
    df = construir(df)
    graficar(df, tok)
    cols = ["mes", "hashprice_usd_ph_dia", "btc_cierre_usd", "h100_usd_hora",
            "mina_vieja_neto", "mina_media_neto", "mina_nueva_neto",
            "hosting", "gpu_neto", "brecha_gpu_vs_mina_media", "brecha_hosting_vs_mina_media"]
    tabla = df[cols].copy()
    for c in cols[1:]:
        tabla[c] = tabla[c].round(0)
    tabla.to_csv(OUT / "tabla_mw.csv", index=False)
    print(tabla.dropna(subset=["mina_media_neto"]).to_string(index=False))
    print("\nCorrelaciones (pearson, n):")
    for k, v in correlaciones(df).items():
        print(f"  {k}: r={v[0]}  n={v[1]}")


    mes  hashprice_usd_ph_dia  btc_cierre_usd  h100_usd_hora  mina_vieja_neto  mina_media_neto  mina_nueva_neto  hosting  gpu_neto  brecha_gpu_vs_mina_media  brecha_hosting_vs_mina_media
2023-09                  62.0             NaN            NaN            940.0           2374.0           3419.0   3899.0       NaN                       NaN                           2.0
2024-08                  44.0         58970.0            NaN            324.0           1336.0           2073.0   3899.0       NaN                       NaN                           3.0
2024-09                  42.0         63330.0            3.0            284.0           1269.0           1987.0   3899.0   28529.0                      22.0                           3.0
2024-10                  47.0         70215.0            NaN            437.0           1526.0           2320.0   3899.0       NaN                       NaN                           3.0
2024-11                  55.0         96449.0            NaN     

## Qué dice (lectura, no dato)

1. **Hosting de IA sin GPU propia rinde 2x a 7x lo que mina una flota 17.5 J/TH**, y la brecha se abrió al caer el hashprice en 2026. Es el contrato que firman TeraWulf y Cipher; no requiere los US$20 M por MW de capex de GPU, solo el capex del data center.
2. **Una flota de 29.5 J/TH a 4.8 c/kWh queda en cero o negativo en 5 de los últimos 8 meses (feb, mar, abr, jun y jul 2026; abr en -2).** Ese es el minero que tiene que decidir algo este año.
3. **H100 propias dan 8x a 34x**, pero con un capex de otro orden (US$20 M vs 0.85 M por MW). La decisión no es "cuál rinde más", es "quién financia el capex", y ese es el estudio.
4. **Fee rate vs precio de inferencia: no hay serie que correlacionar.** Fees bajaron de picos de más de 50% (may 2023) y 72% (abr 2024) del ingreso minero a 0.5-0.8% y ahí se quedaron; tokens nivel GPT-4 cayeron ~90x de mar 2023 a dic 2025. Las dos series solo coinciden en un mes (dic 2025), así que no hay correlación que medir: son dos tendencias con causas distintas.
5. **BTC vs H100 da r = -0.68 con n = 13.** Interesante y sospechoso: ambas series tienen tendencia propia (BTC subió y bajó; H100 bajó y subió), y 13 puntos no prueban nada. Queda como pregunta, no como hallazgo.

## Qué falta
- Hashprice y BTC mensuales de ene-2023 a jul-2024 (Yahoo BTC-USD, Hashrate Index API).
- H100 mensual real: Silicon Data publica rangos, no meses. El índice está en Bloomberg; la API es de pago.
- Fee rate mediana mensual en sat/vB: mempool.space API, por bloque, agregar a mano.
- Costo realizado por byte (hashrate × J/TH × 600 s ÷ bytes del bloque) por inscripción: eso es el siguiente notebook.
